In [ ]:
from datasets import load_dataset, concatenate_datasets
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

import re
import unicodedata

## 1. Data Exploration

In [59]:
ds = load_dataset("pythainlp/wisesight_sentiment")

In [60]:
df = concatenate_datasets([
    ds["train"],
    ds["validation"],
    ds["test"],
])

df = pd.DataFrame(df)

output_dir = Path("../data/raw")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "data.csv"
df.to_csv(output_path)


In [61]:
#0=pos, 1=neu, 2=neg, 3=q

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

Shape: (26703, 2)

Columns:
['texts', 'category']

Data types:
texts       object
category     int64
dtype: object


In [62]:
#Missing Values

missing = df.isnull().sum().sort_values(ascending=False)

display(
    pd.DataFrame({
        "missing_count": missing,
        "missing_pct": missing / len(df) * 100
    })
)

,missing_count,missing_pct
texts,0,0.0
category,0,0.0


In [63]:
#Duplicated Values

print("Duplicated row:", df.duplicated().sum())
print("Duplicated text:", df.duplicated(subset=["texts"]).sum())

display(
    df[df.duplicated(subset=["texts"], keep=False)].sort_values("texts").head(20)
)

Duplicated row: 0
Duplicated text: 24


,texts,category
9278,toyota ผมแนะนำถ้าใครซื้ออย่าเอารถค้างสต็อคตามป...,0
4977,toyota ผมแนะนำถ้าใครซื้ออย่าเอารถค้างสต็อคตามป...,2
13250,ชุมพรมั้ยอ้ะ?!😂,0
7008,ชุมพรมั้ยอ้ะ?!😂,3
15561,ซีวิค มีส่วนลดเท่าไหร่ครับ,0
13932,ซีวิค มีส่วนลดเท่าไหร่ครับ,3
26163,นกแอร์กับแอร์เอเชียเที่ยวบินไประนองยกเลิกมั้ยค่ะ,1
3214,นกแอร์กับแอร์เอเชียเที่ยวบินไประนองยกเลิกมั้ยค่ะ,3
23817,น้องอยากกิน,0
19476,น้องอยากกิน,1


In [64]:
df_processed = df.dropna(subset="texts")
df_processed = df_processed.drop_duplicates(subset="texts")

In [65]:
print("Shape:", df_processed.shape)

print("\nColumns:")
print(df_processed.columns.tolist())

print("\nData types:")
print(df_processed.dtypes)

Shape: (26679, 2)

Columns:
['texts', 'category']

Data types:
texts       object
category     int64
dtype: object


In [66]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "data.csv"
df_processed.to_csv(output_path)


## 2. Data Preprocessing

In [ ]:
input_path = Path("../data/processed/data.csv")

df = pd.read_csv(input_path)
df = df.drop(columns="Unnamed: 0")
df.head(5)

In [ ]:
#clean text
def clean_text(text):
    text = unicodedata.normalize("NFC", str(text))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"@\w+", "USER", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["texts_clean"] = df["texts"].apply(clean_text)

In [ ]:
#sentiment name
LABEL_NAMES = {
    0: "positive",
    1: "neutral",
    2: "negative",
    3: "question"
}

df["sentiment"] = df["category"].map(LABEL_NAMES)
df.head(5)

In [ ]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "data.csv"
df.to_csv(output_path)

## 3. Model Training

In [ ]:
input_path = Path("../data/processed/data.csv")
random_state = 42
test_size = 0.2

In [ ]:
df = pd.read_csv(input_path)

df = df[["texts_clean", "category"]]

In [ ]:
X = df["texts_clean"]
y = df["category"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

In [ ]:
#Model

model = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(2, 5), max_features=100_000, sublinear_tf=True)),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=random_state))
])


In [ ]:
model.fit(X_train, y_train)

## 4. Model Evaluation

In [ ]:
y_pred = model.predict(X_test)

print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1, 2, 3],
        target_names=["positive", "neutral", "negative", "question"]
    )
)

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0,1,2,3])

plt.figure(figsize=(6, 5))

sns.heatmap(cm, annot=True, fmt="d", cmap="coolwarm", xticklabels=["positive", "neutral", "negative", "question"], yticklabels=["positive", "neutral", "negative", "question"])

plt.title("Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()